# Alkahest Browser Export

Exports the merged Kaggle Heretic checkpoint to a browser-ready ONNX package and optionally publishes it to Hugging Face.

In [ ]:
import os
from pathlib import Path

REPO_URL = os.environ.get("HERETIC_TO_ONNX_REPO", "https://github.com/alkahest-ai/heretic-to-onnx.git")
REPO_REF = os.environ.get("HERETIC_TO_ONNX_REF", "codex/kaggle-heretic-2b-run")
SOURCE_REPO_ID = os.environ.get("SOURCE_REPO_ID", "thomasjvu/alkahest-0.8b-heretic-merged")
TARGET_REPO_ID = os.environ.get("TARGET_REPO_ID", "thomasjvu/alkahest-0.8b-q4-webgpu")
BASE_MODEL_ID = os.environ.get("BASE_MODEL_ID", "Qwen/Qwen3.5-0.8B")
TEMPLATE_PATH = os.environ.get("TEMPLATE_PATH", "configs/heretic-to-onnx.qwen3-5-0.8b-heretic-q4-webgpu.yaml")
REPO_DIR = Path("/kaggle/working/heretic-to-onnx")
MANIFEST_PATH = Path("/kaggle/working/alkahest-browser-export.yaml")
WORK_DIR = Path("/kaggle/working/browser-export-work")
PACKAGE_DIR = Path("/kaggle/working/alkahest-browser-package")
print({
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "source_repo_id": SOURCE_REPO_ID,
    "target_repo_id": TARGET_REPO_ID,
    "base_model_id": BASE_MODEL_ID,
    "template_path": TEMPLATE_PATH,
})


In [ ]:
import platform
import shutil
import torch

print("python_platform=", platform.platform())
print("cuda_available=", torch.cuda.is_available())
print("gpu_count=", torch.cuda.device_count())
gpu_names = []
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    gpu_names.append(props.name)
    print(f"gpu_{i}=", props.name, round(props.total_memory / 1024**3, 2), "GiB")
usage = shutil.disk_usage(Path("/kaggle/working"))
print("working_disk_free_gb=", round(usage.free / 1024**3, 2))
expected_gpu = os.environ.get("EXPECTED_GPU_SUBSTRING", "T4")
if expected_gpu and not all(expected_gpu in name for name in gpu_names):
    raise RuntimeError(f"Expected GPU containing '{expected_gpu}', got {gpu_names}")


In [ ]:
!pip install -U --no-cache-dir "transformers @ git+https://github.com/huggingface/transformers.git@c472755e79aac54d675845bff5e5c821c21260af" accelerate huggingface_hub hf_transfer onnx onnxscript onnxruntime-gpu onnxconverter-common sentencepiece safetensors pillow

In [ ]:
!git clone --depth 1 --branch {REPO_REF} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git rev-parse --short HEAD

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    try:
        os.environ.setdefault("HF_TOKEN", secrets.get_secret("HF_TOKEN"))
    except Exception:
        pass
except Exception:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
    print("HF_TOKEN found; publish step will run.")
else:
    print("HF_TOKEN not set; export will run without publish.")


In [ ]:
!python -m tools.heretic_to_onnx render-manifest --template {TEMPLATE_PATH} --output {MANIFEST_PATH} --source-model-id {SOURCE_REPO_ID} --base-model-id {BASE_MODEL_ID} --target-repo-id {TARGET_REPO_ID}
!cat {MANIFEST_PATH}

In [ ]:
publish_flag = "--publish" if os.environ.get("HF_TOKEN") else ""
!python scripts/staged_browser_export.py --config {MANIFEST_PATH} --work-dir {WORK_DIR} --output-dir {PACKAGE_DIR} --python-exec python --export-device cuda --export-torch-dtype float16 --trim-source-weights --delete-raw-onnx {publish_flag}

In [ ]:
!du -sh /kaggle/working/*
!find {PACKAGE_DIR} -maxdepth 2 -type f | sort